# Sample execution

To keep this run practical, the sentiment section uses the first 2,000 reviews and the context-prediction section uses the first 1,000 reviews with one training iteration. The full-dataset code remains in the original and commented notebooks. These outputs are illustrative and are not the full experiment results.


# Download the IMDB Dataset

### My understanding (draft): the review data

The notebook reads movie reviews from `reviews.txt` and their sentiment labels from `labels.txt`. Each review is text, while its matching label says whether it is positive or negative. Keeping the files beside this notebook lets the relative paths in the original code find them.

Before continuing, I should check that both files load and note how many reviews the notebook reports: **[add my result]**.

In [1]:
# Download reviews.txt and labels.txt from here: https://github.com/udacity/deep-learning/tree/master/sentiment-network

def pretty_print_review_and_label(i):
   print(labels[i] + "\t:\t" + reviews[i][:80] + "...")

g = open('reviews.txt','r') # What we know!
reviews = list(map(lambda x:x[:-1],g.readlines()))
g.close()

g = open('labels.txt','r') # What we WANT to know!
labels = list(map(lambda x:x[:-1].upper(),g.readlines()))
g.close()

# Capturing Word Correlation in Input Data

### My understanding (draft): representing words with numbers

A computer cannot directly do arithmetic on words, so the example first represents words as numeric vectors. One-hot vectors assign a separate position to every word; they identify a word but do not express how words relate. An embedding instead gives each word a compact vector whose values can be learned from the task.

In my own words, the key difference I noticed is: **[add my explanation]**.

In [2]:
import numpy as np

onehots = {}
onehots['cat'] = np.array([1,0,0,0])
onehots['the'] = np.array([0,1,0,0])
onehots['dog'] = np.array([0,0,1,0])
onehots['sat'] = np.array([0,0,0,1])

sentence = ['the','cat','sat']
x = onehots[sentence[0]] + \
    onehots[sentence[1]] + \
    onehots[sentence[2]]

print("Sent Encoding:" + str(x))

Sent Encoding:[1 1 0 1]


# Predicting Movie Reviews

### My understanding (draft): learning word embeddings from sentiment

The model learns weights from labeled reviews to predict sentiment. Its input representation includes word-specific vectors, so training can adjust those vectors along with the prediction weights. Words that help distinguish positive from negative reviews can end up with useful learned relationships.

The training code uses a simplified setup and only a small number of iterations. I should record what the output says about its performance: **[add my result]**.

In [3]:
import sys

f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

f = open('labels.txt')
raw_labels = f.readlines()
f.close()

# Quick demonstration: use a small balanced subset.
raw_reviews = raw_reviews[:2000]
raw_labels = raw_labels[:2000]

tokens = list(map(lambda x:set(x.split(" ")),raw_reviews))

vocab = set()
for sent in tokens:
    for word in sent:
        if(len(word)>0):
            vocab.add(word)
vocab = list(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_indices)))

target_dataset = list()
for label in raw_labels:
    if label == 'positive\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

In [4]:
import numpy as np
np.random.seed(1)

def sigmoid(x):
    return 1/(1 + np.exp(-x))

alpha, iterations = (0.01, 2)
hidden_size = 100

weights_0_1 = 0.2*np.random.random((len(vocab),hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,1)) - 0.1

correct,total = (0,0)
for iter in range(iterations):
    
    # train on first 24,000
    for i in range(len(input_dataset)-1000):

        x,y = (input_dataset[i],target_dataset[i])
        layer_1 = sigmoid(np.sum(weights_0_1[x],axis=0)) #embed + sigmoid
        layer_2 = sigmoid(np.dot(layer_1,weights_1_2)) # linear + softmax

        layer_2_delta = layer_2 - y # compare pred with truth
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) #backprop

        weights_0_1[x] -= layer_1_delta * alpha
        weights_1_2 -= np.outer(layer_1,layer_2_delta) * alpha

        if(np.abs(layer_2_delta) < 0.5):
            correct += 1
        total += 1
        if(i % 10 == 9):
            progress = str(i/float(len(input_dataset)))
            sys.stdout.write('\rIter:'+str(iter)\
                             +' Progress:'+progress[2:4]\
                             +'.'+progress[4:6]\
                             +'% Training Accuracy:'\
                             + str(correct/float(total)) + '%')
    print()
correct,total = (0,0)
for i in range(len(input_dataset)-1000,len(input_dataset)):

    x = input_dataset[i]
    y = target_dataset[i]

    layer_1 = sigmoid(np.sum(weights_0_1[x],axis=0))
    layer_2 = sigmoid(np.dot(layer_1,weights_1_2))
    
    if(np.abs(layer_2 - y) < 0.5):
        correct += 1
    total += 1
print("Test Accuracy:" + str(correct / float(total)))

Iter:0 Progress:49.95% Training Accuracy:0.439%
Iter:1 Progress:02.95% Training Accuracy:0.45849056603773586%

Iter:1 Progress:49.95% Training Accuracy:0.6355%


Test Accuracy:0.766


In [5]:
tokens[0]

{'',
 '\n',
 '.',
 'a',
 'about',
 'adults',
 'age',
 'all',
 'and',
 'as',
 'at',
 'believe',
 'bromwell',
 'burn',
 'can',
 'cartoon',
 'classic',
 'closer',
 'comedy',
 'down',
 'episode',
 'expect',
 'far',
 'fetched',
 'financially',
 'here',
 'high',
 'i',
 'immediately',
 'in',
 'insightful',
 'inspector',
 'is',
 'isn',
 'it',
 'knew',
 'lead',
 'life',
 'line',
 'm',
 'many',
 'me',
 'much',
 'my',
 'of',
 'one',
 'other',
 'pathetic',
 'pettiness',
 'pity',
 'pomp',
 'profession',
 'programs',
 'ran',
 'reality',
 'recalled',
 'remind',
 'repeatedly',
 'right',
 's',
 'sack',
 'same',
 'satire',
 'saw',
 'school',
 'schools',
 'scramble',
 'see',
 'situation',
 'some',
 'student',
 'students',
 'such',
 'survive',
 't',
 'teachers',
 'teaching',
 'than',
 'that',
 'the',
 'their',
 'think',
 'through',
 'time',
 'to',
 'tried',
 'welcome',
 'what',
 'when',
 'which',
 'who',
 'whole',
 'years',
 'your'}

# Comparing Word Embeddings

### My understanding (draft): comparing learned word vectors

The comparison code looks for words whose learned vectors are close to a target word?s vector. This gives one way to inspect what the model has encoded. Similarity here is based on the model?s training task and data; it does not mean the words are identical in every context.

Words shown as similar to `beautiful` and `terrible` in my run: **[record examples and what surprised me]**.

In [6]:
from collections import Counter
import math 

def similar(target='beautiful'):
    target_index = word2index[target]
    scores = Counter()
    for word,index in word2index.items():
        raw_difference = weights_0_1[index] - (weights_0_1[target_index])
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))

    return scores.most_common(10)

In [7]:
print(similar('beautiful'))

[('beautiful', -0.0), ('acquaints', -0.6302352833960375), ('sliding', -0.6418895501783313), ('breed', -0.6521585085820478), ('info', -0.6590343145642218), ('famously', -0.6733834408493836), ('leisurely', -0.6759332506035027), ('reminiscent', -0.6800229157040395), ('crunching', -0.6827743362050612), ('alta', -0.6830728112005071)]


In [8]:
print(similar('terrible'))

[('terrible', -0.0), ('ligabue', -0.6313130323982923), ('tile', -0.635591951412486), ('storm', -0.6491192158585098), ('defenders', -0.6609409814484266), ('didn', -0.6674645482087531), ('mouthpiece', -0.6701454271713447), ('predator', -0.6717230621826813), ('misunderstandings', -0.6769881682642926), ('prejudice', -0.6780331625560874)]


# Filling in the Blank

### My understanding (draft): using context to predict a missing word

This section trains embeddings using nearby words as context. The model uses surrounding words to help predict a target word, so a word?s vector is shaped by the contexts in which it appears. This is a different learning signal from predicting only the review sentiment.

I should note what the context window and negative examples do in the code, using my own words: **[add my explanation]**.

In [9]:
import sys,random,math
from collections import Counter
import numpy as np

np.random.seed(1)
random.seed(1)
f = open('reviews.txt')
raw_reviews = f.readlines()[:1000]
f.close()

tokens = list(map(lambda x:(x.split(" ")),raw_reviews))
wordcnt = Counter()
for sent in tokens:
    for word in sent:
        wordcnt[word] -= 1
vocab = list(set(map(lambda x:x[0],wordcnt.most_common())))

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

concatenated = list()
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
            concatenated.append(word2index[word])
        except:
            ""
    input_dataset.append(sent_indices)
concatenated = np.array(concatenated)
random.shuffle(input_dataset)

In [10]:
alpha, iterations = (0.05, 1)
hidden_size,window,negative = (50,2,5)

weights_0_1 = (np.random.rand(len(vocab),hidden_size) - 0.5) * 0.2
weights_1_2 = np.random.rand(len(vocab),hidden_size)*0

layer_2_target = np.zeros(negative+1)
layer_2_target[0] = 1

def similar(target='beautiful'):
  target_index = word2index[target]

  scores = Counter()
  for word,index in word2index.items():
    raw_difference = weights_0_1[index] - (weights_0_1[target_index])
    squared_difference = raw_difference * raw_difference
    scores[word] = -math.sqrt(sum(squared_difference))
  return scores.most_common(10)

def sigmoid(x):
    return 1/(1 + np.exp(-x))

for rev_i,review in enumerate(input_dataset * iterations):
  for target_i in range(len(review)):
        
    # since it's really expensive to predict every vocabulary
    # we're only going to predict a random subset
    target_samples = [review[target_i]]+list(concatenated\
    [(np.random.rand(negative)*len(concatenated)).astype('int').tolist()])

    left_context = review[max(0,target_i-window):target_i]
    right_context = review[target_i+1:min(len(review),target_i+window)]

    layer_1 = np.mean(weights_0_1[left_context+right_context],axis=0)
    layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T))
    layer_2_delta = layer_2 - layer_2_target
    layer_1_delta = layer_2_delta.dot(weights_1_2[target_samples])

    weights_0_1[left_context+right_context] -= layer_1_delta * alpha
    weights_1_2[target_samples] -= np.outer(layer_2_delta,layer_1)*alpha

  if(rev_i % 250 == 0):
    sys.stdout.write('\rProgress:'+str(rev_i/float(len(input_dataset)
        *iterations)) + "   " + str(similar('terrible')))
  sys.stdout.write('\rProgress:'+str(rev_i/float(len(input_dataset)
        *iterations)))
print(similar('terrible'))

Progress:0.006

Progress:0.012

Progress:0.016

Progress:0.023

Progress:0.034

Progress:0.043

... (90 outputs left out)

Progress:0.978

Progress:0.989

Progress:0.999

[('terrible', -0.0), ('predictable', -0.8640891264137893), ('festival', -0.9088372338233823), ('horrible', -0.9904679985084738), ('itself', -0.9919451926529357), ('fiction', -1.0218873866490477), ('roll', -1.0472661016389524), ('japan', -1.0717367999756975), ('completed', -1.0768548963547924), ('entertaining', -1.087282492431646)]


# King - Man + Woman ~= Queen

### My understanding (draft): vector arithmetic analogy

The analogy example combines and compares word vectors to search for a word that best matches a relationship, such as `king - man + woman`. This is an experiment with relationships in the learned vector space, not a guarantee that every analogy will work. The result depends on the training data and learned vectors.

Results from my run and what they suggest: **[add my observations]**.

In [11]:
def analogy(positive=['terrible','good'],negative=['bad']):
    
    norms = np.sum(weights_0_1 * weights_0_1,axis=1)
    norms.resize(norms.shape[0],1)
    
    normed_weights = weights_0_1 * norms
    
    query_vect = np.zeros(len(weights_0_1[0]))
    for word in positive:
        query_vect += normed_weights[word2index[word]]
    for word in negative:
        query_vect -= normed_weights[word2index[word]]
    
    scores = Counter()
    for word,index in word2index.items():
        raw_difference = weights_0_1[index] - query_vect
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
        
    return scores.most_common(10)[1:]

In [12]:
analogy(['terrible','good'],['bad'])

[('good', -33.38970943233201),
 ('great', -33.965869823555),
 ('it', -34.120710483695106),
 ('his', -34.281341898744074),
 ('there', -34.36550362119059),
 ('your', -34.37101257639519),
 ('very', -34.40683480517983),
 ('the', -34.49335077075874),
 ('them', -34.504856462410686)]

In [13]:
analogy(['elizabeth','he'],['she'])

[('it', -97.89390557423486),
 ('film', -98.11717002013127),
 ('movie', -98.57489194219323),
 ('i', -98.68826530898683),
 ('she', -98.80395023928101),
 ('story', -99.43366797420622),
 ('there', -99.45252531973885),
 ('they', -99.65319149663677),
 ('show', -99.7180393504373)]